# An RNN from scratch (NumPy only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudanum/MIA5150/blob/main/notebooks/sequential-models-rnn-gru-lstm/01_rnn_from_scratch.ipynb)

*MIA5150 — Generative AI & LLMs · University of Ottawa · Instructor: Imran Ahmad*

This notebook opens the sequential-models module by building a recurrent neural network with **nothing but NumPy** — no PyTorch, no autograd, no `nn.RNN`. You will write the forward recurrence by hand, derive the gradients (backpropagation through time) by hand, *prove* them correct against finite differences, and then train the network to memorize a repeating text. The point is simple: **there is no magic.** Everything a framework will do for you later in this course is exactly what you will do here, line by line.

**Learning goals**

- Write down the RNN recurrence and explain **weight sharing across time** and the **unrolled** view.
- Implement a character-level forward pass with one-hot inputs and a softmax output.
- Derive **backpropagation through time (BPTT)**, implement it, and validate it with a numerical gradient check.
- Train the network with a hand-rolled **Adam** optimizer and **gradient clipping**.
- **Sample** text from the trained model and control randomness with **temperature**.
- See first-hand where vanilla RNNs struggle — the motivation for GRUs and LSTMs (notebook 02).

## 1 · The recurrence

A recurrent neural network reads a sequence **one element at a time**, carrying a **hidden state** $h_t$ — a fixed-size vector that acts as the network's memory of everything it has read so far. At each time step $t$:

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h)$$
$$y_t = W_{hy}\, h_t + b_y$$

- $x_t$ is the input at step $t$. We work at the **character level**, so $x_t$ is a **one-hot vector** of size $V$ (the vocabulary): all zeros except a single 1 at the index of the current character.
- $h_t \in \mathbb{R}^{H}$ is the hidden state; before reading anything, $h_0 = \mathbf{0}$.
- $y_t \in \mathbb{R}^{V}$ is a vector of **logits** — one score per possible next character. A **softmax** turns the logits into a probability distribution over the vocabulary:

$$p_t = \mathrm{softmax}(y_t), \qquad p_t[i] = \frac{e^{y_t[i]}}{\sum_{j} e^{y_t[j]}}$$

For **next-character prediction**, the target at step $t$ is simply the character at step $t+1$: the whole training signal is "predict what comes next".

### Weight sharing across time

The crucial idea: **the same matrices $W_{xh}, W_{hh}, W_{hy}$ are used at every time step.** The network does not own separate weights per position — it learns *one* transition rule and applies it everywhere. This is why an RNN can process sequences of any length, and why its parameter count does not grow with the sequence.

### The "unrolled" view

Copy the cell once per time step and wire $h_{t-1} \to h_t$; the RNN becomes a **deep feedforward network of depth $T$ with shared weights**, an input injected at every layer and an output read off every layer:

```
        x_1          x_2          x_3
         │            │            │
         ▼            ▼            ▼
 h_0 ──▶ ( tanh ) ──▶ ( tanh ) ──▶ ( tanh ) ──▶ h_3
         │   W_hh     │   W_hh     │
         ▼            ▼            ▼
        y_1          y_2          y_3
```

Every `( tanh )` block is the *same* cell with the *same* weights. Training this unrolled network is exactly what **backpropagation through time** does in section 4.

In [ ]:
# ============================================================
# Config — every knob you might want to turn lives here
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

SEED        = 42
HIDDEN_SIZE = 32      # number of units in the hidden state h_t
SEQ_LEN     = 25      # characters per training window
STEPS       = 2000    # training iterations
LR          = 0.01    # Adam learning rate
CLIP        = 5.0     # gradient-norm clipping threshold
REPEATS     = 10      # how many times the phrase repeats in the corpus

np.random.seed(SEED)

def one_hot(i, n):
    """One-hot vector of length n with a 1 at position i."""
    v = np.zeros(n)
    v[i] = 1.0
    return v

def softmax(logits):
    """Numerically stable softmax over a 1-D array of logits."""
    z = logits - np.max(logits)   # subtracting the max changes nothing mathematically
    e = np.exp(z)                 # but keeps exp() from overflowing
    return e / e.sum()

print(f"NumPy {np.__version__} — no deep-learning framework anywhere in sight.")

## 2 · The forward pass, in code

The class below holds the five parameter arrays and runs the recurrence. Two things to watch as you read it:

- `forward` takes a **list of one-hot vectors** — one per time step — and returns the hidden states, the logits, and a **cache** of per-step intermediates. The cache exists because backpropagation (section 5) will need $x_t$, $h_{t-1}$ and $h_t$ at every step.
- Weights are initialized **small** (`* 0.1`): large random weights would push the pre-activations into the flat tails of $\tanh$, where gradients are near zero and learning stalls before it starts.

In [ ]:
class RNNNumpy:
    """A single-layer vanilla RNN, implemented with nothing but NumPy.

        h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)
        y_t = W_hy h_t + b_y          (logits over the vocabulary)
    """

    def __init__(self, vocab_size, hidden_size, seed=SEED):
        rng = np.random.default_rng(seed)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.W_xh = rng.standard_normal((hidden_size, vocab_size)) * 0.1  # input  -> hidden
        self.W_hh = rng.standard_normal((hidden_size, hidden_size)) * 0.1 # hidden -> hidden (the memory)
        self.W_hy = rng.standard_normal((vocab_size, hidden_size)) * 0.1  # hidden -> logits
        self.b_h = np.zeros(hidden_size)
        self.b_y = np.zeros(vocab_size)

    def forward(self, inputs, h0=None):
        """Run the recurrence over a sequence.

        inputs : list of one-hot vectors, one per time step
        h0     : initial hidden state (zeros if not given)

        Returns (hs, ys, cache):
          hs    : (T, hidden_size) array — the hidden state after each step
          ys    : (T, vocab_size) array  — the logits after each step
          cache : per-step (x_t, h_(t-1), h_t), needed later by backprop
        """
        h = np.zeros(self.hidden_size) if h0 is None else h0
        hs, ys, cache = [], [], []
        for x in inputs:
            h_prev = h
            h = np.tanh(self.W_xh @ x + self.W_hh @ h_prev + self.b_h)
            y = self.W_hy @ h + self.b_y
            hs.append(h)
            ys.append(y)
            cache.append((x, h_prev, h))
        return np.array(hs), np.array(ys), cache

    def params(self):
        """All trainable parameters, by name (the same dict the optimizer will update)."""
        return {"W_xh": self.W_xh, "W_hh": self.W_hh,
                "W_hy": self.W_hy, "b_h": self.b_h, "b_y": self.b_y}


# A quick look at the shapes, for a toy 10-character alphabet:
probe = RNNNumpy(vocab_size=10, hidden_size=HIDDEN_SIZE)
n_params = sum(p.size for p in probe.params().values())
for name, p in probe.params().items():
    print(f"{name:5s} {str(p.shape):10s}")
print(f"\nonly {n_params} parameters in total — weight sharing keeps it tiny")

## 3 · A first look inside an (untrained) net

Before any training, let's just *run* the recurrence over a short string and look at the hidden states. In the heatmap below, **each row is one $h_t$** (after reading the character shown on the left) and **each column is one hidden unit**. The weights are random, so nothing here is learned yet — that is precisely the point.

> **Every $h_t$ is a trace of everything seen so far.** It is the only channel through which the past can influence the next prediction. Right now that trace is meaningless noise; after training it will come to encode things like "we are in the middle of the word `quick`".

In [ ]:
demo_text = "the quick brown fox"
demo_chars = sorted(set(demo_text))
demo_stoi = {c: i for i, c in enumerate(demo_chars)}
demo_inputs = [one_hot(demo_stoi[c], len(demo_chars)) for c in demo_text]

demo_rnn = RNNNumpy(vocab_size=len(demo_chars), hidden_size=HIDDEN_SIZE, seed=0)
hs, ys, _ = demo_rnn.forward(demo_inputs)

vmax = np.abs(hs).max()
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(hs, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_yticks(range(len(demo_text)))
ax.set_yticklabels([f"{t}: '{c}'" for t, c in enumerate(demo_text)], fontsize=8)
ax.set_xlabel("hidden unit")
ax.set_ylabel("time step (character just read)")
ax.set_title("Hidden state $h_t$ over time — an untrained RNN")
fig.colorbar(im, ax=ax, label="tanh activation (bounded in −1..1)")
plt.tight_layout()
plt.show()

**What to notice**

- The hidden state changes at *every* step — reading the same character twice (the two `'o'`s in `brown`) produces **different** rows, because $h_{t-1}$ differs. State = input *plus* history, never input alone.
- Activations stay bounded — that is $\tanh$ squashing the pre-activation into $(-1, 1)$, which keeps the state from blowing up as the sequence grows.
- With an untrained net there is no obvious structure: this is what "noise" looks like before learning. Keep this picture in mind — after training, the same plot would show clear, repeatable patterns locked to the phrase.

## 4 · Backpropagation through time (BPTT)

**Setup.** We train on next-character prediction: at every step $t$ the target $y_t^*$ is the *next* character. The loss is the **mean cross-entropy over all $T$ timesteps**,

$$L = \frac{1}{T} \sum_{t=1}^{T} L_t, \qquad L_t = -\log p_t[y_t^*], \qquad p_t = \mathrm{softmax}(y_t).$$

**Output layer.** For softmax + cross-entropy, the gradient w.r.t. the logits is famously simple:

$$\frac{\partial L}{\partial y_t} = \frac{1}{T}\,(p_t - \mathbf{1}_{y_t^*})$$

where $\mathbf{1}_{y_t^*}$ is the one-hot target. From $y_t = W_{hy} h_t + b_y$ the parameter gradients follow:

$$\frac{\partial L}{\partial W_{hy}} \mathrel{+}= \frac{\partial L}{\partial y_t}\, h_t^\top, \qquad \frac{\partial L}{\partial b_y} \mathrel{+}= \frac{\partial L}{\partial y_t}$$

**The hidden state receives gradient from TWO places.** $h_t$ influences the loss through its own output $y_t$ **and** through the next hidden state $h_{t+1}$, which had $h_t$ as an input. Both contributions must be added:

$$\frac{\partial L}{\partial h_t} = W_{hy}^\top \frac{\partial L}{\partial y_t} \;+\; \underbrace{W_{hh}^\top \, \frac{\partial L}{\partial z_{t+1}}}_{\text{flowing back from step } t+1}$$

**Through the tanh.** With $h_t = \tanh(z_t)$ and $\tanh'(z) = 1 - \tanh^2(z)$:

$$\frac{\partial L}{\partial z_t} = \frac{\partial L}{\partial h_t} \odot \bigl(1 - h_t^2\bigr)$$

(We can write $1 - h_t^2$ instead of $1 - \tanh^2(z_t)$ — this is why the forward pass caches $h_t$.)

**Parameter gradients accumulate over time.** Because $W_{xh}, W_{hh}, b_h$ are *shared* across all steps, every step contributes to the *same* gradient — this is weight sharing seen from the backward side:

$$\frac{\partial L}{\partial W_{xh}} = \sum_t \frac{\partial L}{\partial z_t}\, x_t^\top, \qquad
\frac{\partial L}{\partial W_{hh}} = \sum_t \frac{\partial L}{\partial z_t}\, h_{t-1}^\top, \qquad
\frac{\partial L}{\partial b_h} = \sum_t \frac{\partial L}{\partial z_t}$$

**The sweep.** Initialize the "gradient from the future" to zero and walk $t = T, T-1, \dots, 1$, carrying it backward — exactly like backprop through a deep feedforward net, except the weight matrices are the same at every "layer".

> **Why gradients vanish (teaser).** The gradient reaching step $t$ from far in the future is a *product* of many terms of the form $W_{hh}^\top \mathrm{diag}(1 - h^2)$. Since $|1 - \tanh^2| \le 1$, these long products typically shrink exponentially — the reason vanilla RNNs forget long-range structure, and the entire motivation for notebook 02 (GRU & LSTM).

## 5 · `loss_and_grads` — forward + full BPTT, in code

The function below runs the forward pass, computes the mean cross-entropy, and sweeps backward exactly as derived above. We pass the network in explicitly so that the *same* function can later be called on a tiny second network for the gradient check. Compare each line with the derivation in section 4 — there is a one-to-one correspondence.

In [ ]:
def loss_and_grads(rnn, xs, ys):
    """Forward pass + full backpropagation through time.

    xs : list of T one-hot input vectors
    ys : list of T target indices (the next character at each step)

    Returns (loss, grads) with grads keyed exactly like rnn.params().
    """
    T = len(xs)
    hs, logits, cache = rnn.forward(xs)

    # ---- forward: mean cross-entropy over timesteps ----
    loss, probs = 0.0, []
    for t in range(T):
        p = softmax(logits[t])
        probs.append(p)
        loss -= np.log(p[ys[t]] + 1e-12)
    loss /= T

    # ---- backward: sweep t = T-1 ... 0 ----
    grads = {k: np.zeros_like(p) for k, p in rnn.params().items()}
    dh_next = np.zeros(rnn.hidden_size)   # no future beyond the last step

    for t in reversed(range(T)):
        x, h_prev, h = cache[t]

        dy = probs[t].copy()              # d(softmax cross-entropy)/d(logits)
        dy[ys[t]] -= 1.0                  #   = p - onehot(target)
        dy /= T                           # the loss is the MEAN over t

        grads["W_hy"] += np.outer(dy, h)
        grads["b_y"] += dy

        dh = rnn.W_hy.T @ dy + dh_next    # gradient from y_t AND from h_{t+1}
        dz = dh * (1.0 - h * h)           # tanh'(z) = 1 - tanh(z)^2, and tanh(z) = h

        grads["W_xh"] += np.outer(dz, x)
        grads["W_hh"] += np.outer(dz, h_prev)
        grads["b_h"] += dz

        dh_next = rnn.W_hh.T @ dz         # passed one more step into the past

    return loss, grads

### Proving it correct: the numerical gradient check

Hand-derived backprop is notoriously easy to get *subtly* wrong — a missing transpose or a forgotten $\odot\,(1-h^2)$ still *runs*, it just trains badly. The standard antidote is to compare every analytic gradient against a **finite-difference** estimate, which knows nothing about your derivation:

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \epsilon) - L(\theta_i - \epsilon)}{2\epsilon}, \qquad
\text{rel. error} = \frac{|g_{\text{analytic}} - g_{\text{numeric}}|}{\max\!\bigl(10^{-8},\; |g_{\text{analytic}}| + |g_{\text{numeric}}|\bigr)}$$

On a tiny config (**vocab 5, hidden 4, seq 5**) we can afford to check **every single parameter**. If the max relative error comes out below $10^{-5}$, the backprop is right — the `assert` below *is* the unit test, and this pattern is worth reusing any time you hand-roll gradients.

In [ ]:
# Tiny config: vocab 5, hidden 4, seq 5 — small enough to check EVERY parameter.
V_TINY, H_TINY, T_TINY = 5, 4, 5
check_rng = np.random.default_rng(0)
tiny = RNNNumpy(vocab_size=V_TINY, hidden_size=H_TINY, seed=7)
seq = check_rng.integers(0, V_TINY, size=T_TINY + 1)
xs = [one_hot(i, V_TINY) for i in seq[:-1]]
ys = list(seq[1:])

_, analytic = loss_and_grads(tiny, xs, ys)

EPS = 1e-5
max_rel_err, worst = 0.0, None
for name, param in tiny.params().items():
    grad = analytic[name]
    it = np.nditer(param, flags=["multi_index"])
    while not it.finished:
        ix = it.multi_index
        old = param[ix]
        param[ix] = old + EPS
        loss_up, _ = loss_and_grads(tiny, xs, ys)
        param[ix] = old - EPS
        loss_dn, _ = loss_and_grads(tiny, xs, ys)
        param[ix] = old                        # restore exactly
        numeric = (loss_up - loss_dn) / (2 * EPS)
        rel = abs(numeric - grad[ix]) / max(1e-8, abs(numeric) + abs(grad[ix]))
        if rel > max_rel_err:
            max_rel_err, worst = rel, (name, ix)
        it.iternext()

n_checked = sum(p.size for p in tiny.params().values())
print(f"checked all {n_checked} parameters (central differences, eps = {EPS})")
print(f"max relative error: {max_rel_err:.3e}   worst at {worst}")
assert max_rel_err < 1e-5, "BPTT and finite differences disagree — the backprop is wrong!"
print("gradient check PASSED — the backprop derivation is correct")

## 6 · The learning task: a repeating corpus

The task: memorize a short corpus made of **one repeating phrase** —

`the quick brown fox jumps over the lazy dog. ` × 10

Why a *repeating* corpus? Because it is (almost) deterministic: given enough recent context there is exactly **one** correct next character, so a perfect model could drive the loss close to zero. To do that, the network must discover the cycle and encode "where am I in the phrase" inside its hidden state.

Training details:

- **Windows**: each step samples a random window of `SEQ_LEN` characters; the targets are the same window **shifted by one** (next-char prediction).
- **Gradient clipping** at norm 5.0: BPTT gradients flow through long products of $W_{hh}$ and can explode; rescaling the whole gradient when its norm exceeds the threshold is the standard cheap fix.
- **Adam**, hand-rolled in about 15 lines — first/second moment estimates with bias correction — so that nothing about the optimizer is hidden either.
- The dashed line on the plot is the **uniform-guess baseline** $\ln V$: the loss of a model that always predicts a flat distribution over the vocabulary. Anything below it means the model is genuinely learning structure.

In [ ]:
phrase = "the quick brown fox jumps over the lazy dog. "
text = phrase * REPEATS

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = [stoi[c] for c in text]   # the whole corpus as integer ids

print(f"corpus length : {len(text)} chars  ({REPEATS} repeats of a {len(phrase)}-char phrase)")
print(f"vocab size    : {vocab_size}")
print(f"vocab         : {''.join(chars)!r}")
print(f"first 90 chars: {text[:90]!r}")

In [ ]:
class Adam:
    """A minimal Adam optimizer (Kingma & Ba, 2015) — the whole algorithm in ~15 lines."""

    def __init__(self, params, lr=LR, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = {k: np.zeros_like(p) for k, p in params.items()}
        self.v = {k: np.zeros_like(p) for k, p in params.items()}
        self.t = 0

    def step(self, grads):
        self.t += 1
        for k, p in self.params.items():
            g = grads[k]
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g        # first moment (mean of grads)
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * g * g    # second moment (mean of grads^2)
            m_hat = self.m[k] / (1 - self.b1 ** self.t)                # bias correction
            v_hat = self.v[k] / (1 - self.b2 ** self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


def clip_grads(grads, max_norm):
    """Rescale all gradients together if their global norm exceeds max_norm."""
    norm = np.sqrt(sum(np.sum(g * g) for g in grads.values()))
    if norm > max_norm:
        for k in grads:
            grads[k] *= max_norm / (norm + 1e-12)
    return norm


rnn = RNNNumpy(vocab_size, HIDDEN_SIZE, seed=SEED)
opt = Adam(rnn.params())          # params() returns REFERENCES, so Adam updates rnn in place
losses, norms = [], []

for step in range(STEPS):
    start = np.random.randint(0, len(data) - SEQ_LEN - 1)
    window = data[start : start + SEQ_LEN + 1]
    xs = [one_hot(i, vocab_size) for i in window[:-1]]
    targets = window[1:]                    # next-char prediction: shifted by one

    loss, grads = loss_and_grads(rnn, xs, targets)
    norms.append(clip_grads(grads, CLIP))
    opt.step(grads)
    losses.append(loss)

    if step % 400 == 0 or step == STEPS - 1:
        print(f"step {step:5d} | loss {loss:.4f} | grad norm {norms[-1]:8.3f}")

baseline = np.log(vocab_size)     # loss of an "always guess 1/V" predictor
smooth_w = 50
smoothed = np.convolve(losses, np.ones(smooth_w) / smooth_w, mode="valid")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, alpha=0.25, label="loss (per step)")
ax.plot(np.arange(smooth_w - 1, STEPS), smoothed, linewidth=2, label=f"loss (smoothed, w={smooth_w})")
ax.axhline(baseline, ls="--", color="gray", label=f"uniform-guess baseline ln(V) ≈ {baseline:.2f}")
ax.set_xlabel("training step")
ax.set_ylabel("mean cross-entropy")
ax.set_title("Training loss — next-character prediction on the repeating corpus")
ax.legend()
plt.tight_layout()
plt.show()

**What to notice**

- The loss starts right around the uniform-guess baseline $\ln V$ — an untrained net's best strategy really is "predict everything equally" — and then drops **clearly below it**, so the model is learning the cycle rather than guessing.
- The raw curve is noisy step to step because each window is a different slice of the corpus; watch the smoothed line for the trend.
- Gradient norms are largest early on (the net is badly wrong everywhere) and settle as the loss falls — clipping catches the occasional spike without stalling learning.

## 7 · Sampling: before and after

To **generate** text, feed a seed character, sample the next character from the softmax, feed it back in as the next input, and repeat. `sample(seed, n, temperature)` does exactly that.

**Temperature** $\tau$ rescales the logits before the softmax, $p = \mathrm{softmax}(y / \tau)$: $\tau < 1$ sharpens the distribution (safe, repetitive), $\tau > 1$ flattens it (more random, more mistakes). Below we sample from a fresh net built with the **same seed** — i.e. the exact weights the trained net started from — and then from the trained net.

In [ ]:
def sample(rnn, seed_char, n, temperature=1.0):
    """Generate n characters by feeding each prediction back as the next input.

    temperature < 1 sharpens the distribution (conservative, repetitive);
    temperature > 1 flattens it (more random, more mistakes).
    """
    x = one_hot(stoi[seed_char], vocab_size)
    h = np.zeros(rnn.hidden_size)
    out = seed_char
    for _ in range(n):
        h = np.tanh(rnn.W_xh @ x + rnn.W_hh @ h + rnn.b_h)
        p = softmax((rnn.W_hy @ h + rnn.b_y) / temperature)
        i = np.random.choice(vocab_size, p=p)
        out += itos[i]
        x = one_hot(i, vocab_size)
    return out


# BEFORE: a fresh net with the SAME seed = exactly the weights rnn started from.
untrained = RNNNumpy(vocab_size, HIDDEN_SIZE, seed=SEED)
print("BEFORE training (temperature 1.0):")
print(sample(untrained, "t", 120, temperature=1.0))
print()
print("AFTER training (temperature 0.8):")
print(sample(rnn, "t", 180, temperature=0.8))

**What to notice**

- **BEFORE**, the output is near-uniform noise over the vocabulary — the untrained net's best guess is "anything".
- **AFTER**, the model reproduces the phrase cleanly and loops it. It has essentially *memorized the cycle* — it cannot say anything else, which is expected for a corpus this tiny. Generalization comes from data scale, and that is the story of the rest of the course.
- This also shows what a char-level RNN really is: a very good **local pattern matcher**. Nothing here "understands" the sentence — and yet the recurrence is all it takes to capture the pattern.

## Try it yourself

1. **Capacity: hidden size 8 vs 64.** Set `HIDDEN_SIZE = 8` in the Config cell and re-run from the corpus cell down; then try `64`. This corpus is tiny, so even 8 units can fit it — the interesting difference is *how fast* each converges and how the final losses compare.
2. **A much longer cycle.** Replace the corpus with a ~200-character pattern repeated 5 times (e.g. concatenate several different sentences into one long cycle) and keep `SEQ_LEN = 25`. You should see the loss plateau noticeably higher and the samples break down first at the transitions that depend on older context. Why does the vanilla RNN struggle when the pattern gets long? **Notebook 02 (GRU & LSTM) is the answer.**
3. **Temperature sweep.** Sample 200 characters from the trained net at `temperature` 0.5, 1.0 and 1.5. Describe what changes and connect it to what temperature does to the softmax distribution.
4. **Break the gradient check on purpose.** In `loss_and_grads`, delete the `dy /= T` line and re-run the gradient-check cell — the assertion should now fail. In one sentence: why must the mean-loss factor appear in the analytic gradient for it to match finite differences of the mean loss?

## Recap

- An RNN is **one rule applied repeatedly**: $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$, $y_t = W_{hy} h_t + b_y$, with the **same weights shared across every time step**.
- The hidden state is the network's *entire* memory of the past — every prediction is a function of $h_t$ alone.
- **BPTT** is just the chain rule on the unrolled graph: each $h_t$ receives gradient from its own output *and* from the next hidden state, and parameter gradients are **accumulated over all timesteps** because of weight sharing.
- A finite-difference **gradient check** (max relative error < $10^{-5}$) is the cheapest way to *prove* hand-written backprop correct — reach for it whenever you derive gradients yourself.
- Gradient **clipping** keeps BPTT stable; a 15-line **Adam** trains the net; **temperature** controls sampling randomness.
- Vanilla RNNs degrade on long-range patterns because gradients vanish through long products of $W_{hh}$ — next up, **notebook 02: GRU and LSTM**, which exist precisely to fix that.